In [1]:
from huggingface_hub import login
import os
login(os.getenv("HF_TOKEN"))

In [ ]:
# This notebook is for testing a smaller dataset and training configuration to make sure everything works before training on the full dataset

from datasets import load_dataset
from transformers import BeitForImageClassification, AutoImageProcessor, TrainingArguments, Trainer #changed from BeitFeatureExtractor to BeitFeatureProcessor, because depricated
import evaluate
import numpy as np
import torch

model_name = "microsoft/beit-base-patch16-224"
dataset = load_dataset("D123aniel/560_test_dataset")

feature_extractor = AutoImageProcessor.from_pretrained(model_name)
num_labels = len(dataset["train"].features["label"].names)

model = BeitForImageClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label={i: label for i, label in enumerate(dataset["train"].features["label"].names)},
    label2id={label: i for i, label in enumerate(dataset["train"].features["label"].names)},
    ignore_mismatched_sizes=True
)

# Convert to BEiT tensor
def transform(example):
    inputs = feature_extractor(images=example["image"], return_tensors="pt")
    return {"pixel_values": inputs["pixel_values"][0], "label": example["label"]}

encoded_dataset = dataset.map(transform, batched=False)

training_args = TrainingArguments(
    output_dir="./beit_finetuned",
    per_device_train_batch_size=8, #switched from 32 to 8 for testing
    per_device_eval_batch_size=8, #switched from 32 to 8 for testing
    gradient_accumulation_steps=2, #changed from 1
    num_train_epochs=5,
    learning_rate=3e-5,
    warmup_ratio=0.05,
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    bf16=True,
    dataloader_num_workers=4,
    report_to="tensorboard",
    seed=42,
)

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return accuracy_metric.compute(predictions=preds, references=p.label_ids)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

trainer.train()

You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

BeitForImageClassification LOAD REPORT from: microsoft/beit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


True
NVIDIA GeForce RTX 5070 Ti


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.542196,0.666667
2,No log,0.273529,1.000000
3,No log,0.120920,1.000000
4,No log,0.071761,1.000000
5,No log,0.065103,1.000000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=25, training_loss=0.3976152038574219, metrics={'train_runtime': 54.8493, 'train_samples_per_second': 6.563, 'train_steps_per_second': 0.456, 'total_flos': 2.788519270957056e+16, 'train_loss': 0.3976152038574219, 'epoch': 5.0})

In [10]:
pred_output = trainer.predict(encoded_dataset["test"])
preds = np.argmax(pred_output.predictions, axis=1)
test_accuracy = (preds == pred_output.label_ids).mean()
print("Test accuracy:", float(test_accuracy))

Test accuracy: 0.875
